# 05 - Model Interpretation

This notebook examines the trained models to identify the key features that drive income prediction. Using model-specific interpretation methods (e.g., coefficients, feature importance, SHAP values), it analyzes how different variables contribute to the models' predictions and extracts the most influential predictors of income.

## Imports

In [1]:
import sys

import numpy as np
import pandas as pd
import sklearn.metrics

from IPython.display import display, HTML

sys.path.append("../src")

from my_datasets import census
from my_datasets import dataframe_utils
import ml_utils
from ml_utils import feature_importance
from ml_utils import evaluation
from ml_utils import model_pipelines

## Helper Functions

### `train_full_model`

Reusable utility function that loads a previously trained model from disk if available. If no saved model exists, it trains a new model using the full dataset, saves it for future reuse, and then returns the fitted model.

In [2]:
def train_full_model(
    model_name,
    df,
    categorical_columns,
    numeric_columns,
    target_column,
    weight_column,
    model_filename_prefix="key_drivers"
):
    model_filename = f"{model_filename_prefix}-{model_name}.joblib"
    try:
        pipeline = census.load_model(model_filename)
        print(f"Loaded pretrained {model_name} model.")
    except FileNotFoundError:
        print(f"No pretrained {model_name} model found.")
        model = model_pipelines.get_prediction_models(
        	categorical_columns=categorical_columns,
        	numeric_columns=numeric_columns,
            remove_niu=True,
            select=model_name
        )
        
        import time
        from datetime import timedelta
        
        start = time.perf_counter()
        pipeline = evaluation.fit_model(
        	estimator=model,
        	X=data_df[categorical_columns + numeric_columns],
        	y=data_df[target_column],
        	sample_weight=(
                data_df[weight_column] / data_df[weight_column].mean()
            )
        )
        end = time.perf_counter()
        
        elapsed = end - start
        print(f"  {timedelta(seconds=elapsed)}")
    
        census.save_model(pipeline, model_filename)

    return pipeline

## Feature Exclusion

Several features have previously been selected for exclusion from the models. This section defines those features.

In [3]:
exclude_features = [
    "hispanic origin",
    "detailed occupation recode",
    "detailed industry recode",

    "veterans benefits",
    "detailed household and family stat",

    "family members under 18",
    "region of previous residence",
    "state of previous residence",

    "migration code-change in msa",
    "migration code-change in reg",
    "migration code-move within reg",
    "migration prev res in sunbelt",

    "year"
]

## Data Loading and Inspection

The cleaned, processed dataset is loaded into a dataframe and basic inspection is performed to verify that it has been loaded correctly.

The target and sample weight columns are identified, along with the numeric-categorical features.

In [4]:
data_df = census.load_processed_dataframe("clean_data.csv", header=0, verbose=False)

dataframe_utils.print_dataframe_info(data_df)

target_column = census.get_target_feature()
weight_column = census.get_sample_weights_feature()

numeric_categorical_features = census.get_numeric_categorical_features()

# # subsample for code testing
# from sklearn.model_selection import train_test_split
# data_df, _ = train_test_split(
#     data_df,
#     train_size=1000,
#     stratify=data_df[target_column],
#     random_state=42,
# )
# data_df = data_df.reset_index(drop=True)
# dataframe_utils.print_dataframe_info(data_df)

DataFrame Memory Usage: 362.04 MB


,dtype,count,non_null,null_count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,int64,199523,199523,0,NaN,NaN,NaN,34.494199,22.310895,0.0,15.0,33.0,50.0,90.0
class of worker,str,199523,199523,0,9,Not in universe,100245,NaN,NaN,NaN,NaN,NaN,NaN,NaN
detailed industry recode,int64,199523,199523,0,NaN,NaN,NaN,15.35232,18.067129,0.0,0.0,0.0,33.0,51.0
detailed occupation recode,int64,199523,199523,0,NaN,NaN,NaN,11.306556,14.454204,0.0,0.0,0.0,26.0,46.0
education,str,199523,199523,0,17,High school graduate,48407,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wage per hour,int64,199523,199523,0,NaN,NaN,NaN,55.426908,274.896454,0.0,0.0,0.0,0.0,9999.0
enroll in edu inst last wk,str,199523,199523,0,3,Not in universe,186943,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital stat,str,199523,199523,0,7,Never married,86485,NaN,NaN,NaN,NaN,NaN,NaN,NaN
major industry code,str,199523,199523,0,24,Not in universe or children,100684,NaN,NaN,NaN,NaN,NaN,NaN,NaN
major occupation code,str,199523,199523,0,15,Not in universe,100684,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,age,class of worker,detailed industry recode,detailed occupation recode,education,wage per hour,enroll in edu inst last wk,marital stat,major industry code,major occupation code,race,hispanic origin,sex,member of a labor union,reason for unemployment,full or part time employment stat,capital gains,capital losses,dividends from stocks,tax filer stat,region of previous residence,state of previous residence,detailed household and family stat,detailed household summary in household,weight,migration code-change in msa,migration code-change in reg,migration code-move within reg,live in this house 1 year ago,migration prev res in sunbelt,num persons worked for employer,family members under 18,country of birth father,country of birth mother,country of birth self,citizenship,own business or self employed,fill inc questionnaire for veteran's admin,veterans benefits,weeks worked in year,year,label
0,73,Not in universe,0,0,High school graduate,0,Not in universe,Widowed,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Not in labor force,0,0,0,Nonfiler,Not in universe,Not in universe,Other Rel 18+ ever marr not in subfamily,Other relative of householder,1700.09,NaN,NaN,NaN,Not in universe under 1 year old,NaN,0,Not in universe,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,0,95,- 50000.
1,58,Self-employed-not incorporated,4,34,Some college but no degree,0,Not in universe,Divorced,Construction,Precision production craft & repair,White,All other,Male,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Head of household,South,Arkansas,Householder,Householder,1053.55,MSA to MSA,Same county,Same county,No,Yes,1,Not in universe,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,52,94,- 50000.
2,18,Not in universe,0,0,10th grade,0,High school,Never married,Not in universe or children,Not in universe,Asian or Pacific Islander,All other,Female,Not in universe,Not in universe,Not in labor force,0,0,0,Nonfiler,Not in universe,Not in universe,Child 18+ never marr Not in a subfamily,Child 18 or older,991.95,NaN,NaN,NaN,Not in universe under 1 year old,NaN,0,Not in universe,Vietnam,Vietnam,Vietnam,Foreign born- Not a citizen of U S,0,Not in universe,2,0,95,- 50000.
3,9,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Nonfiler,Not in universe,Not in universe,Child <18 never marr not in subfamily,Child under 18 never married,1758.14,Nonmover,Nonmover,Nonmover,Yes,Not in universe,0,Both parents present,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.
4,10,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Nonfiler,Not in universe,Not in universe,Child <18 never marr not in subfamily,Child under 18 never married,1069.16,Nonmover,Nonmover,Nonmover,Yes,Not in universe,0,Both parents present,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199518,87,Not in universe,0,0,7th and 8th grade,0,Not in universe,Married-civilian spouse present,Not in universe or children,Not in universe,White,All other,Male,Not in universe,Not in universe,Not in labor force,0,0,0,Joint both 65+,Not in universe,Not in universe,Householder,Householder,955.27,NaN,NaN,NaN,Not in universe under 1 year old,NaN,0,Not in universe,Canada,United-States,United-States,Native- Born in the United States,0,Not in universe,2,0,95,- 50000.
199519,65,Self-employed-incorporated,37,2,11t

## Feature Type Separation and Adjustments

The dataset features are split into categorical and numeric column groups, with additional numeric-categorical features explicitly treated as categorical.

Adjustments are made to ensure correct feature assignment and prevent data leakage. The target column is removed from the categorical feature list, while the sample weight column is removed from the numeric feature list. In addition, all preselected excluded features are removed from consideration.

In [5]:
categorical_columns, numeric_columns = ml_utils.get_categorical_numeric_split(
    data_df,
    treat_as_categorical=numeric_categorical_features
)

exclude_set = set(exclude_features)
exclude_set.update([target_column, weight_column])

categorical_columns = [
    column
    for column in categorical_columns
    if column not in exclude_set
]

numeric_columns = [
    column
    for column in numeric_columns
    if column not in exclude_set
]

## Model Interpretation

In this section, the selected models are trained on the full processed dataset to identify the features most strongly associated with income. The resulting models and interpretation outputs are saved for subsequent analysis.

### LASSO Logistic Regression

In [6]:
pipeline = train_full_model(
    "lasso_logistic_regression",
    data_df,
    categorical_columns,
    numeric_columns,
    target_column,
    weight_column
)
display(pipeline)

importance_df = feature_importance.logistic_regression_importance(pipeline)
census.save_results_dataframe(
    importance_df, filename=f"key_drivers-lasso_logistic_regression.csv"
)

with pd.option_context(
	"display.max_colwidth", None,
	"display.max_columns", None,
	"display.width", None
):
    feature_importance.display_logistic_regression_importance(importance_df)

Loaded pretrained lasso_logistic_regression model.


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('LogisticRegression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the

2026-06-24 15:39:28,901 - INFO - Saved DataFrame to results directory: /Users/jagraves21/census-income-ml-project/data/Census/results/key_drivers-lasso_logistic_regression.csv


,feature_raw,feature_type,feature_name,feature_value,coefficient,abs_coefficient,odds_ratio
0,categorical__education_Doctorate degree(PhD EdD),categorical,education,Doctorate degree(PhD EdD),2.113870,2.113870,8.280220
1,categorical__education_Prof school degree (MD DDS DVM LLB JD),categorical,education,Prof school degree (MD DDS DVM LLB JD),2.013194,2.013194,7.487191
2,categorical__tax filer stat_Nonfiler,categorical,tax filer stat,Nonfiler,-1.702619,1.702619,0.182206
3,categorical__sex_Female,categorical,sex,Female,-1.466858,1.466858,0.230649
4,categorical__education_Masters degree(MA MS MEng MEd MSW MBA),categorical,education,Masters degree(MA MS MEng MEd MSW MBA),1.295326,1.295326,3.652188
...,...,...,...,...,...,...,...
252,categorical__country of birth self_Panama,categorical,country of birth self,Panama,0.000000,0.000000,1.000000
253,categorical__country of birth self_Portugal,categorical,country of birth self,Portugal,0.000000,0.000000,1.000000
254,categorical__country of birth self_Thailand,categorical,country of birth self,Thailand,0.000000,0.000000,1.000000
255,categorical__country of birth self_Vietnam,categorical,country of birth self,Vietnam,0.000000,0.000000,1.000000


,feature_raw,feature_type,feature_name,feature_value,coefficient,abs_coefficient,odds_ratio
0,categorical__education_Doctorate degree(PhD EdD),categorical,education,Doctorate degree(PhD EdD),2.113870,2.113870,8.280220
1,categorical__education_Prof school degree (MD DDS DVM LLB JD),categorical,education,Prof school degree (MD DDS DVM LLB JD),2.013194,2.013194,7.487191
4,categorical__education_Masters degree(MA MS MEng MEd MSW MBA),categorical,education,Masters degree(MA MS MEng MEd MSW MBA),1.295326,1.295326,3.652188
6,numeric__weeks worked in year,numeric,weeks worked in year,NaN,1.116011,1.116011,3.052653
11,categorical__major occupation code_Executive admin and managerial,categorical,major occupation code,Executive admin and managerial,0.946532,0.946532,2.576758


,feature_raw,feature_type,feature_name,feature_value,coefficient,abs_coefficient,odds_ratio
2,categorical__tax filer stat_Nonfiler,categorical,tax filer stat,Nonfiler,-1.702619,1.702619,0.182206
3,categorical__sex_Female,categorical,sex,Female,-1.466858,1.466858,0.230649
5,categorical__education_10th grade,categorical,education,10th grade,-1.139901,1.139901,0.319851
7,categorical__major industry code_Social services,categorical,major industry code,Social services,-1.055915,1.055915,0.347874
8,categorical__education_5th or 6th grade,categorical,education,5th or 6th grade,-1.045202,1.045202,0.351621


,feature_group,mean_abs_coefficient
0,categorical,61.456557
1,numeric,2.793752


,feature_type,feature_name,mean_abs_coefficient
0,numeric,weeks worked in year,1.116011
1,categorical,sex,0.898035
2,categorical,education,0.827111
3,numeric,age,0.713407
4,categorical,own business or self employed,0.576206
5,categorical,major occupation code,0.518401
6,categorical,tax filer stat,0.477588
7,numeric,capital gains,0.454175
8,numeric,dividends from stocks,0.349415
9,categorical,enroll in edu inst last wk,0.342183


### Ridge Logistic Regression

In [7]:
pipeline = train_full_model(
    "ridge_logistic_regression",
    data_df,
    categorical_columns,
    numeric_columns,
    target_column,
    weight_column
)
display(pipeline)

importance_df = feature_importance.logistic_regression_importance(pipeline)
census.save_results_dataframe(
    importance_df, filename=f"key_drivers-ridge_logistic_regression.csv"
)

with pd.option_context(
	"display.max_colwidth", None,
	"display.max_columns", None,
	"display.width", None
):
    feature_importance.display_logistic_regression_importance(importance_df)

Loaded pretrained ridge_logistic_regression model.


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('LogisticRegression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the

2026-06-24 15:39:29,023 - INFO - Saved DataFrame to results directory: /Users/jagraves21/census-income-ml-project/data/Census/results/key_drivers-ridge_logistic_regression.csv


,feature_raw,feature_type,feature_name,feature_value,coefficient,abs_coefficient,odds_ratio
0,categorical__education_Doctorate degree(PhD EdD),categorical,education,Doctorate degree(PhD EdD),2.177892,2.177892,8.827680
1,categorical__education_Prof school degree (MD DDS DVM LLB JD),categorical,education,Prof school degree (MD DDS DVM LLB JD),2.096429,2.096429,8.137060
2,categorical__tax filer stat_Nonfiler,categorical,tax filer stat,Nonfiler,-1.766543,1.766543,0.170923
3,categorical__sex_Female,categorical,sex,Female,-1.503386,1.503386,0.222376
4,categorical__education_5th or 6th grade,categorical,education,5th or 6th grade,-1.423451,1.423451,0.240881
...,...,...,...,...,...,...,...
252,categorical__country of birth self_Portugal,categorical,country of birth self,Portugal,0.012605,0.012605,1.012685
253,numeric__wage per hour,numeric,wage per hour,NaN,-0.008246,0.008246,0.991788
254,categorical__country of birth self_Hungary,categorical,country of birth self,Hungary,-0.006383,0.006383,0.993638
255,categorical__detailed household summary in household_Child under 18 ever married,categorical,detailed household summary in household,Child under 18 ever married,-0.003438,0.003438,0.996568


,feature_raw,feature_type,feature_name,feature_value,coefficient,abs_coefficient,odds_ratio
0,categorical__education_Doctorate degree(PhD EdD),categorical,education,Doctorate degree(PhD EdD),2.177892,2.177892,8.827680
1,categorical__education_Prof school degree (MD DDS DVM LLB JD),categorical,education,Prof school degree (MD DDS DVM LLB JD),2.096429,2.096429,8.137060
5,categorical__education_Masters degree(MA MS MEng MEd MSW MBA),categorical,education,Masters degree(MA MS MEng MEd MSW MBA),1.362621,1.362621,3.906419
6,numeric__weeks worked in year,numeric,weeks worked in year,NaN,1.102071,1.102071,3.010395
10,categorical__country of birth self_Scotland,categorical,country of birth self,Scotland,1.016819,1.016819,2.764388


,feature_raw,feature_type,feature_name,feature_value,coefficient,abs_coefficient,odds_ratio
2,categorical__tax filer stat_Nonfiler,categorical,tax filer stat,Nonfiler,-1.766543,1.766543,0.170923
3,categorical__sex_Female,categorical,sex,Female,-1.503386,1.503386,0.222376
4,categorical__education_5th or 6th grade,categorical,education,5th or 6th grade,-1.423451,1.423451,0.240881
7,categorical__education_9th grade,categorical,education,9th grade,-1.072956,1.072956,0.341996
8,categorical__education_1st 2nd 3rd or 4th grade,categorical,education,1st 2nd 3rd or 4th grade,-1.053289,1.053289,0.348789


,feature_group,mean_abs_coefficient
0,categorical,83.035751
1,numeric,2.769060


,feature_type,feature_name,mean_abs_coefficient
0,numeric,weeks worked in year,1.102071
1,categorical,sex,0.931119
2,categorical,education,0.902364
3,numeric,age,0.708416
4,categorical,own business or self employed,0.620746
5,categorical,major occupation code,0.549173
6,categorical,tax filer stat,0.498243
7,categorical,enroll in edu inst last wk,0.477674
8,numeric,capital gains,0.449972
9,categorical,race,0.372448


### Decision Tree

In [8]:
pipeline = train_full_model(
    "decision_tree",
    data_df,
    categorical_columns,
    numeric_columns,
    target_column,
    weight_column
)
display(pipeline)

importance_df = feature_importance.decision_tree_importance(pipeline)
census.save_results_dataframe(
    importance_df, filename=f"key_drivers-decision_tree.csv"
)

with pd.option_context(
	"display.max_colwidth", None,
	"display.max_columns", None,
	"display.width", None
):
    feature_importance.display_decision_tree_importance(importance_df)

Loaded pretrained decision_tree model.


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('DecisionTreeClassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

2026-06-24 15:39:29,147 - INFO - Saved DataFrame to results directory: /Users/jagraves21/census-income-ml-project/data/Census/results/key_drivers-decision_tree.csv


,feature_raw,feature_type,feature_name,feature_value,gini_importance
0,numeric__age,numeric,age,NaN,0.126628
1,numeric__capital gains,numeric,capital gains,NaN,0.126373
2,numeric__dividends from stocks,numeric,dividends from stocks,NaN,0.099272
3,numeric__weeks worked in year,numeric,weeks worked in year,NaN,0.071253
4,categorical__sex_Male,categorical,sex,Male,0.031487
5,numeric__capital losses,numeric,capital losses,NaN,0.026177
6,categorical__major occupation code_Professional specialty,categorical,major occupation code,Professional specialty,0.022069
7,categorical__major occupation code_Executive admin and managerial,categorical,major occupation code,Executive admin and managerial,0.020292
8,numeric__wage per hour,numeric,wage per hour,NaN,0.013190
9,categorical__live in this house 1 year ago_Yes,categorical,live in this house 1 year ago,Yes,0.013050


,feature_group,mean_gini_importance
0,categorical,0.537107
1,numeric,0.462893


,feature_type,feature_name,mean_gini_importance
0,numeric,age,0.126628
1,numeric,capital gains,0.126373
2,numeric,dividends from stocks,0.099272
3,numeric,weeks worked in year,0.071253
4,numeric,capital losses,0.026177
5,categorical,sex,0.017294
6,numeric,wage per hour,0.013190
7,categorical,live in this house 1 year ago,0.009353
8,categorical,num persons worked for employer,0.006759
9,categorical,own business or self employed,0.006342


### XGBoost

In [9]:
pipeline = train_full_model(
    "xgboost",
    data_df,
    categorical_columns,
    numeric_columns,
    target_column,
    weight_column
)
display(pipeline)

importance_df = feature_importance.xgboost_importance(
    pipeline, data_df[categorical_columns + numeric_columns]
)
census.save_results_dataframe(
    importance_df, filename=f"key_drivers-xgboost.csv"
)

with pd.option_context(
	"display.max_colwidth", None,
	"display.max_columns", None,
	"display.width", None
):
    feature_importance.display_xgboost_importance(importance_df)

Loaded pretrained xgboost model.


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('XGBStringClassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of th

2026-06-24 15:40:00,333 - INFO - Saved DataFrame to results directory: /Users/jagraves21/census-income-ml-project/data/Census/results/key_drivers-xgboost.csv


,feature_raw,feature_type,feature_name,feature_value,mean_abs_shap,mean_shap,positive_ratio
0,numeric__weeks worked in year,numeric,weeks worked in year,NaN,0.909304,-0.449554,0.394962
1,numeric__age,numeric,age,NaN,0.894676,-0.622140,0.473785
2,categorical__tax filer stat_Nonfiler,categorical,tax filer stat,Nonfiler,0.598825,-0.457938,0.623632
3,categorical__sex_Female,categorical,sex,Female,0.303162,-0.081856,0.478837
4,numeric__dividends from stocks,numeric,dividends from stocks,NaN,0.234979,-0.100083,0.103281
5,numeric__capital gains,numeric,capital gains,NaN,0.143841,-0.063823,0.020629
6,categorical__education_Bachelors degree(BA AB BS),categorical,education,Bachelors degree(BA AB BS),0.143535,-0.063166,0.099798
7,categorical__detailed household summary in household_Householder,categorical,detailed household summary in household,Householder,0.140654,-0.064475,0.378142
8,categorical__num persons worked for employer_6,categorical,num persons worked for employer,6,0.130913,-0.057791,0.183172
9,categorical__major occupation code_Executive admin and managerial,categorical,major occupation code,Executive admin and managerial,0.118272,-0.047846,0.062624


,feature_group,mean_mean_abs_shap
0,categorical,2.934445
1,numeric,2.280688


,feature_type,feature_name,mean_mean_abs_shap
0,numeric,weeks worked in year,0.909304
1,numeric,age,0.894676
2,numeric,dividends from stocks,0.234979
3,categorical,sex,0.173020
4,numeric,capital gains,0.143841
5,categorical,tax filer stat,0.106438
6,numeric,capital losses,0.061031
7,numeric,wage per hour,0.036856
8,categorical,num persons worked for employer,0.036039
9,categorical,major occupation code,0.032150
